# VAE transaction-anomaly evaluation

This notebook evaluates a Variational Autoencoder on retail
transactions made by me. Flagged transactions require review and do not establish fraud.
The same dataset and splits will be used for RealNVP.

In [1]:
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import sklearn

project = Path("/content/transaction_comparison")

for folder in ["data", "models", "results", "outputs"]:
    (project / folder).mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Select a T4 GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Project folder:", project)

GPU: Tesla T4
PyTorch: 2.11.0+cu130
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1
Project folder: /content/transaction_comparison


## Synthetic transaction data

Transactions are produced specifically for a controlled experiment rather than being derived from real world data or
company records. Features represent basket quantity, item price, discount,
hour and transaction duration.

Training contains normal examples only. Validation and test sets contain
normal and injected unusual examples. Labels and anomaly types are excluded
from model inputs. Results derived from synthetic transaction data fail to substantiate actual fraud performance in real-world scenarios.


In [2]:
import json
import hashlib
from sklearn.preprocessing import StandardScaler

data_seed = 2026
rng = np.random.default_rng(data_seed)

features = [
    "quantity",
    "unit_price",
    "discount",
    "hour_sin",
    "hour_cos",
    "duration_seconds",
]

def make_transactions(n, anomaly_fraction, split):
    # Typical shopping patterns
    quantity = rng.choice(
        [1, 2, 3, 4, 5, 6],
        size=n,
        p=[0.40, 0.30, 0.16, 0.08, 0.04, 0.02],
    ).astype(float)

    unit_price = np.clip(
        rng.lognormal(mean=1.5, sigma=0.4, size=n),
        1.0,
        20.0,
    )

    discount = rng.choice(
        [0.0, 0.05, 0.10, 0.15],
        size=n,
        p=[0.75, 0.10, 0.10, 0.05],
    )

    hour = rng.uniform(6, 22, size=n)

    duration = np.clip(
        15 + 5 * quantity + rng.normal(0, 5, size=n),
        5,
        None,
    )

    labels = np.zeros(n, dtype=int)
    types = np.full(n, "normal", dtype=object)

    unusual_count = int(n * anomaly_fraction)
    unusual_ids = rng.choice(n, size=unusual_count, replace=False)

    # Three predefined unusual patterns
    groups = np.array_split(unusual_ids, 3)

    ids = groups[0]
    quantity[ids] = rng.integers(8, 16, size=len(ids))
    duration[ids] = np.clip(
        15 + 5 * quantity[ids] + rng.normal(0, 5, len(ids)),
        5,
        None,
    )
    types[ids] = "large_quantity"

    ids = groups[1]
    discount[ids] = rng.uniform(0.35, 0.70, size=len(ids))
    types[ids] = "large_discount"

    ids = groups[2]
    quantity[ids] = rng.choice([4.0, 5.0, 6.0], size=len(ids))
    duration[ids] = rng.uniform(5, 12, size=len(ids))
    types[ids] = "short_duration_for_quantity"

    labels[unusual_ids] = 1

    table = pd.DataFrame({
        "transaction_id": [f"{split}_{i:05d}" for i in range(n)],
        "quantity": quantity,
        "unit_price": unit_price,
        "discount": discount,
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "duration_seconds": duration,
        "label": labels,
        "anomaly_type": types,
    })

    return table

# Independently generated splits
train = make_transactions(6000, 0.0, "train")
validation = make_transactions(2000, 0.10, "validation")
test = make_transactions(2000, 0.10, "test")

splits = {
    "train": train,
    "validation": validation,
    "test": test,
}

for name, table in splits.items():
    table.to_csv(project / "data" / f"{name}.csv", index=False)

# Fit preprocessing on training data only
scaler = StandardScaler()
scaler.fit(train[features])

arrays = {}

for name, table in splits.items():
    values = scaler.transform(table[features]).astype(np.float32)
    assert np.isfinite(values).all()
    arrays[f"X_{name}"] = values
    arrays[f"y_{name}"] = table["label"].to_numpy(dtype=np.int64)

np.savez_compressed(
    project / "data" / "shared_arrays.npz",
    **arrays,
)

metadata = {
    "dataset_type": "synthetic controlled experiment",
    "data_seed": data_seed,
    "features": features,
    "training_normal_only": True,
    "validation_anomaly_fraction": 0.10,
    "test_anomaly_fraction": 0.10,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "threshold_plan": (
        "95th percentile of normal validation scores; "
        "flag scores strictly greater than the threshold"
    ),
    "model_seeds": [42, 123, 2026],
    "limitations": [
        "Injected anomalies follow predefined rules.",
        "Some anomaly types are deliberately easy to separate.",
        "Validation and test share the same generating process.",
        "No evidence of performance on real transactions.",
    ],
}

(project / "data" / "dataset_config.json").write_text(
    json.dumps(metadata, indent=2)
)

# Fingerprint the files used by both models
manifest = {}

for path in sorted((project / "data").iterdir()):
    if path.is_file():
        manifest[path.name] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

(project / "data" / "manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(table),
        "normal": int(table["label"].eq(0).sum()),
        "unusual": int(table["label"].eq(1).sum()),
    }
    for name, table in splits.items()
])

display(summary)
print("Model features:", features)
print("Saved shared dataset and preprocessing.")

,split,rows,normal,unusual
0,train,6000,6000,0
1,validation,2000,1800,200
2,test,2000,1800,200


Model features: ['quantity', 'unit_price', 'discount', 'hour_sin', 'hour_cos', 'duration_seconds']
Saved shared dataset and preprocessing.


## VAE training

The VAE learns from normal training examples using reconstruction loss
and KL regularisation.


Architecture and training settings are fixed across three seeds.
Each alert threshold is the 95th percentile of normal validation scores.

In [3]:
import random
import time
import platform
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda")

vae_settings = {
    "input_dim": len(features),
    "hidden_dim": 32,
    "latent_dim": 3,
    "beta": 0.1,
    "epochs": 50,
    "batch_size": 128,
    "learning_rate": 0.001,
    "seeds": [42, 123, 2026],
    "score": "mean squared reconstruction error using latent mean",
}

class TransactionVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        hidden = self.encoder(x)
        return self.mu(hidden), self.logvar(hidden).clamp(-10, 10)

    def forward(self, x):
        mu, logvar = self.encode(x)
        noise = torch.randn_like(mu)
        z = mu + torch.exp(0.5 * logvar) * noise
        return self.decoder(z), mu, logvar

    def reconstruct(self, x):
        mu, _ = self.encode(x)
        return self.decoder(mu)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print("VAE defined.")

VAE defined.


In [4]:
arrays = np.load(project / "data" / "shared_arrays.npz")

X_train = arrays["X_train"]
X_validation = arrays["X_validation"]
y_validation = arrays["y_validation"]

assert np.all(arrays["y_train"] == 0)

train_tensor = torch.from_numpy(X_train)
validation_tensor = torch.from_numpy(X_validation).to(device)

training_records = []

for seed in vae_settings["seeds"]:
    set_seed(seed)

    model = TransactionVAE(
        vae_settings["input_dim"],
        vae_settings["hidden_dim"],
        vae_settings["latent_dim"],
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=vae_settings["learning_rate"],
    )

    loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=vae_settings["batch_size"],
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )

    history = []

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    for epoch in range(1, vae_settings["epochs"] + 1):
        model.train()
        totals = np.zeros(3)

        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            reconstruction, mu, logvar = model(batch)

            reconstruction_loss = (
                (reconstruction - batch).pow(2).mean()
            )

            kl_loss = -0.5 * (
                1 + logvar - mu.pow(2) - logvar.exp()
            ).sum(dim=1).mean()

            loss = reconstruction_loss + vae_settings["beta"] * kl_loss
            assert torch.isfinite(loss).item(), "Non-finite loss."

            loss.backward()
            optimizer.step()

            totals += np.array([
                loss.item(),
                reconstruction_loss.item(),
                kl_loss.item(),
            ]) * len(batch)

        averages = totals / len(train_tensor)

        history.append({
            "epoch": epoch,
            "loss": averages[0],
            "reconstruction_loss": averages[1],
            "kl_loss": averages[2],
        })

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Seed {seed} | epoch {epoch:02d} | "
                f"loss {averages[0]:.4f}",
                flush=True,
            )

    torch.cuda.synchronize()
    training_seconds = time.perf_counter() - start
    training_peak = torch.cuda.max_memory_allocated() / 1024**3

    # Select threshold using normal validation examples only
    model.eval()

    with torch.inference_mode():
        reconstruction = model.reconstruct(validation_tensor)
        validation_scores = (
            (reconstruction - validation_tensor)
            .pow(2)
            .mean(dim=1)
            .cpu()
            .numpy()
        )

    assert np.isfinite(validation_scores).all()

    threshold = float(np.quantile(
        validation_scores[y_validation == 0],
        0.95,
    ))

    torch.save(
        {
            "state_dict": {
                key: value.detach().cpu()
                for key, value in model.state_dict().items()
            },
            "settings": vae_settings,
            "seed": seed,
            "threshold": threshold,
        },
        project / "models" / f"vae_seed{seed}.pt",
    )

    pd.DataFrame(history).to_csv(
        project / "results" / f"vae_history_seed{seed}.csv",
        index=False,
    )

    pd.DataFrame({
        "transaction_id": validation["transaction_id"],
        "label": y_validation,
        "score": validation_scores,
        "prediction": (validation_scores > threshold).astype(int),
    }).to_csv(
        project / "outputs" / f"vae_validation_seed{seed}.csv",
        index=False,
    )

    training_records.append({
        "seed": seed,
        "training_seconds": training_seconds,
        "training_peak_allocated_gpu_gib": training_peak,
        "threshold": threshold,
        "parameters": sum(p.numel() for p in model.parameters()),
    })

    pd.DataFrame(training_records).to_csv(
        project / "results" / "vae_training_summary.csv",
        index=False,
    )

    del model, optimizer, reconstruction
    torch.cuda.empty_cache()

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "precision": "float32",
}

(project / "results" / "vae_config.json").write_text(
    json.dumps({
        "settings": vae_settings,
        "environment": environment,
        "training_timing_scope": (
            "Training loop only, including batch transfers and loss logging; "
            "excludes model setup, validation and file saving."
        ),
        "threshold_method": (
            "95th percentile of normal validation reconstruction errors; "
            "alert when score is strictly greater."
        ),
    }, indent=2)
)

display(pd.DataFrame(training_records).round(4))
print("Three trained models saved. Test set not yet evaluated.")

Seed 42 | epoch 01 | loss 1.0053
Seed 42 | epoch 10 | loss 0.6077
Seed 42 | epoch 20 | loss 0.5693
Seed 42 | epoch 30 | loss 0.5580
Seed 42 | epoch 40 | loss 0.5526
Seed 42 | epoch 50 | loss 0.5481
Seed 123 | epoch 01 | loss 1.0117
Seed 123 | epoch 10 | loss 0.6026
Seed 123 | epoch 20 | loss 0.5673
Seed 123 | epoch 30 | loss 0.5603
Seed 123 | epoch 40 | loss 0.5548
Seed 123 | epoch 50 | loss 0.5506
Seed 2026 | epoch 01 | loss 1.0049
Seed 2026 | epoch 10 | loss 0.6155
Seed 2026 | epoch 20 | loss 0.5709
Seed 2026 | epoch 30 | loss 0.5619
Seed 2026 | epoch 40 | loss 0.5554
Seed 2026 | epoch 50 | loss 0.5465


,seed,training_seconds,training_peak_allocated_gpu_gib,threshold,parameters
0,42,8.7168,0.017,0.4452,2860
1,123,8.9075,0.017,0.3979,2860
2,2026,8.9823,0.017,0.3583,2860


Three trained models saved. Test set not yet evaluated.


### Test evaluation
The saved VAE models are evaluated on the shared test set.
Thresholds were selected using normal validation transactions and remain fixed.
Higher reconstruction error indicates a more unusual transaction.

In [6]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

project = Path("/content/transaction_comparison")
device = torch.device("cuda")

arrays = np.load(project / "data/shared_arrays.npz")
test = pd.read_csv(project / "data/test.csv")

X_test = arrays["X_test"]
y_test = arrays["y_test"].astype(int)

assert len(test) == len(X_test) == len(y_test)

test_tensor = torch.tensor(
    X_test, dtype=torch.float32, device=device
)

test_records = []

for seed in vae_settings["seeds"]:
    checkpoint = torch.load(
        project / f"models/vae_seed{seed}.pt",
        map_location="cpu",
        weights_only=True,
    )

    model = TransactionVAE(
        input_dim=vae_settings["input_dim"],
        hidden_dim=vae_settings["hidden_dim"],
        latent_dim=vae_settings["latent_dim"],
    ).to(device)

    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    threshold = float(checkpoint["threshold"])


    with torch.inference_mode():
        _ = model.reconstruct(test_tensor[:32])
    del _

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    with torch.inference_mode():
        reconstructed = model.reconstruct(test_tensor)
        scores_gpu = (
            (reconstructed - test_tensor).square().mean(dim=1)
        )

    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - start
    peak_gib = torch.cuda.max_memory_allocated() / 1024**3

    scores = scores_gpu.cpu().numpy()
    assert np.isfinite(scores).all()

    predictions = (scores > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    test_records.append({
        "model": "VAE",
        "seed": seed,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(
            y_test, scores
        ),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "false_positive_rate": fp / (fp + tn),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "test_batch_seconds": inference_seconds,
        "test_peak_allocated_gpu_gib": peak_gib,
    })

    predictions_table = test.copy()
    predictions_table["anomaly_score"] = scores
    predictions_table["predicted_unusual"] = predictions

    predictions_table.to_csv(
        project / f"outputs/vae_test_seed{seed}.csv",
        index=False,
    )

    print(f"Evaluated seed {seed}")

    del model, reconstructed, scores_gpu
    torch.cuda.empty_cache()

vae_test_results = pd.DataFrame(test_records)

vae_test_results.to_csv(
    project / "results/vae_test_metrics.csv",
    index=False,
)

display(vae_test_results.round(4))

metrics = [
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
]

display(
    vae_test_results[metrics]
    .agg(["mean", "std"])
    .round(4)
)

print("Saved test predictions and metrics for all three seeds.")

Evaluated seed 42
Evaluated seed 123
Evaluated seed 2026


,model,seed,threshold,roc_auc,average_precision,precision,recall,f1,false_positive_rate,tn,fp,fn,tp,test_batch_seconds,test_peak_allocated_gpu_gib
0,VAE,42,0.4452,0.9923,0.9551,0.7344,0.940,0.8246,0.0378,1732,68,12,188,0.0006,0.0175
1,VAE,123,0.3979,0.9922,0.9486,0.6797,0.955,0.7942,0.0500,1710,90,9,191,0.0004,0.0175
2,VAE,2026,0.3583,0.9889,0.9368,0.7220,0.935,0.8148,0.0400,1728,72,13,187,0.0005,0.0175


,roc_auc,average_precision,precision,recall,f1,false_positive_rate
mean,0.9911,0.9468,0.7120,0.9433,0.8112,0.0426
std,0.0020,0.0093,0.0287,0.0104,0.0155,0.0065


Saved test predictions and metrics for all three seeds.


### VAE findings
When tested across three seeds, the VAE model achieved an average ROC-AUC score of 0.9911.
The model achieved an average precision of 0.9468 and an F1 score of 0.8112.

The average recall was 94.33%, and the false-positive rate was 4.26%.
Some regular transactions would therefore need unnecessary checks.

These results are related to the synthetic transactions and the injected anomaly types used here. They do not determine performance based on actual retail data.
Timing measures a batch of 2,000 transactions run on a warmed GPU and excludes
preprocessing,transfers and model loading.

In [7]:
import shutil
from google.colab import files

# Save the experiment files
backup = shutil.make_archive(
    "/content/pair3_vae_backup",
    "zip",
    root_dir="/content/transaction_comparison",
)

files.download(backup)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>